In [21]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

In [22]:
chla_oan_gems = gpd.read_file("../data/data/chla_oan_gems_dedup.geojson")

print(chla_oan_gems.shape)
chla_oan_gems.head()

(5738, 21)


,index_right,estacion,Decision,param,value,unit,depth,granularidad,fuente,dist,...,id_estacion,nro_muestra,departamento,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,0,URY00006,si,Chl-a,0.0,mg/l,0.3,DIA,GEMS,0.0,...,NaN,NaN,None,None,None,None,None,NaN,None,POINT (649386.009 6180413.463)
1,1,URY00008,dudoso,Chl-a,0.0,mg/l,0.3,DIA,GEMS,0.0,...,NaN,NaN,None,None,None,None,None,NaN,None,POINT (664852.016 6189770.245)
2,1736,XSLH030.S,None,None,NaN,None,NaN,None,OAN,0.0,...,100201.0,25650.0,FLORIDA,CloA_(lab),µg/L,0.400,None,0.1,0.400,POINT (570470.059 6220119.303)
3,1737,XSLH030.S,None,None,NaN,None,NaN,None,OAN,0.0,...,100201.0,25907.0,FLORIDA,CloA_(lab),µg/L,0.700,None,0.1,0.700,POINT (570470.059 6220119.303)
4,1733,XSLH030.S,None,None,NaN,None,NaN,None,OAN,0.0,...,100201.0,26124.0,FLORIDA,CloA_(lab),µg/L,<LD,0.600,1.5,<LD,POINT (570470.059 6220119.303)


In [23]:
oan_registros_clean = gpd.read_file("../data/data/oan/oan_registros_clean.geojson")

print(oan_registros_clean.shape)
oan_registros_clean.head()

(5186, 13)


,nombre_programa,estacion,id_estacion,nro_muestra,departamento,fecha_hora,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry
0,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29873.0,RÍO NEGRO,2019-09-05 16:00:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (-57.41304 -32.9058)
1,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,29985.0,RÍO NEGRO,2019-10-31 09:40:00,CloA_(lab),µg/L,LD<x<LC,0.700000000,2.2,LD<x<LC,POINT (-57.41304 -32.9058)
2,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31062.0,RÍO NEGRO,2020-06-04 14:20:00,CloA_(lab),µg/L,LD<X<LC,0.700000000,2.2,LD<X<LC,POINT (-57.41304 -32.9058)
3,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31410.0,RÍO NEGRO,2020-08-06 12:46:00,CloA_(lab),µg/L,33.000000000,0.700000000,2.2,33.000000000,POINT (-57.41304 -32.9058)
4,Agua Arroyo Grande del Norte DCA,XGRN100.S,100626,31851.0,RÍO NEGRO,2020-11-19 12:29:00,CloA_(lab),µg/L,2.600000000,0.700000000,2.2,2.600000000,POINT (-57.41304 -32.9058)


In [4]:
gems_chla = gpd.read_file("../data/data/joins/gems_chla.geojson")

print(gems_chla.shape)
gems_chla.head()

(3796, 9)


,estacion,Decision,param,fecha,value,unit,depth,granularidad,geometry
0,URY00006,si,Chl-a,2018-11-12 23:20:00,0.0000,mg/l,0.3,DIA,POINT (-55.3727 -34.5071)
1,URY00008,dudoso,Chl-a,2018-11-12 08:30:00,0.0000,mg/l,0.3,DIA,POINT (-55.2061 -34.4204)
2,URY00029,si,Chl-a,2015-11-18 11:53:00,0.0030,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)
3,URY00029,si,Chl-a,2016-01-20 11:20:00,0.0036,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)
4,URY00029,si,Chl-a,2016-10-12 11:10:00,0.0015,mg/l,0.3,DIA,POINT (-56.2355 -34.15748)


### Cuantas mediciones de OAN y GEMS quedaron afuera del dedup?

`chla_oan_gems_dedup.geojson` solo tiene las mediciones de las estaciones que estan compartidas entre OAN y GEMS (ver oan_gems_normalization_v2.ipynb). Las mediciones de estaciones exclusivas de una sola fuente no entraron ahi. Vamos a contarlas.

In [24]:
oan_registros_clean["fuente"] = "OAN"
gems_chla["fuente"] = "GEMS"

oan_registros_clean = oan_registros_clean.rename(columns={"fecha_hora": "fecha"})
gems_chla = gems_chla[pd.to_datetime(gems_chla["fecha"]).dt.year >= 2017].copy()

oan_utm = oan_registros_clean.to_crs(32721).copy()
gems_utm = gems_chla.to_crs(32721).copy()

In [25]:
oan_stations = gpd.GeoDataFrame(geometry=oan_utm["geometry"].unique(), crs=oan_utm.crs)
gems_stations = gpd.GeoDataFrame(geometry=gems_utm["geometry"].unique(), crs=gems_utm.crs)

print(oan_stations.shape, gems_stations.shape)

(359, 1) (153, 1)


In [26]:
oan_match = gpd.sjoin_nearest(oan_utm, gems_stations, how="left", max_distance=0.1, distance_col="dist")
gems_match = gpd.sjoin_nearest(gems_utm, oan_stations, how="left", max_distance=0.1, distance_col="dist")

print(oan_match.shape, gems_match.shape)

(5186, 16) (3086, 12)


In [27]:
oan_exclusivas = oan_match[oan_match["index_right"].isna()].copy()
gems_exclusivas = gems_match[gems_match["index_right"].isna()].copy()

print("Mediciones OAN en estaciones no compartidas con GEMS:", oan_exclusivas.shape[0])
print("Mediciones GEMS en estaciones no compartidas con OAN:", gems_exclusivas.shape[0])

Mediciones OAN en estaciones no compartidas con GEMS: 935
Mediciones GEMS en estaciones no compartidas con OAN: 14


In [28]:
oan_exclusivas["fuente"].value_counts()
gems_exclusivas["fuente"].value_counts()

print("Total OAN:", oan_registros_clean.shape[0], "| en dedup:", (chla_oan_gems["fuente"] == "OAN").sum(), "| exclusivas (afuera):", oan_exclusivas.shape[0])
print("Total GEMS:", gems_chla.shape[0], "| en dedup:", (chla_oan_gems["fuente"] == "GEMS").sum(), "| exclusivas (afuera):", gems_exclusivas.shape[0])

Total OAN: 5186 | en dedup: 3434 | exclusivas (afuera): 935
Total GEMS: 3086 | en dedup: 2304 | exclusivas (afuera): 14


### Unir todo en un solo geojson, ordenado por fecha

Cargamos `chla_oan_gems_dedup.geojson` (estaciones compartidas, ya deduplicado) y le sumamos las mediciones exclusivas de OAN y de GEMS que quedaron afuera. El dedup no tiene columna `fecha` (se perdio al guardarlo, quedo como indice), asi que esas filas van a aparecer sin fecha al ordenar.

In [29]:
union_completo = gpd.GeoDataFrame(
    pd.concat([chla_oan_gems, oan_exclusivas, gems_exclusivas], ignore_index=True),
    geometry="geometry",
    crs=oan_utm.crs
)
union_completo = union_completo.sort_values("fecha").reset_index(drop=True)

print(union_completo.shape)
union_completo["fuente"].value_counts()

(6687, 22)


fuente
OAN     4369
GEMS    2318
Name: count, dtype: int64

In [17]:
union_completo.head()

,index_right,estacion,Decision,param,value,unit,depth,granularidad,fuente,dist,...,departamento,nombre_clave,uni_nombre,valor_original,limite_deteccion,limite_cuantificacion,valor_transformado,geometry,fecha_hora,fecha
0,NaN,URY00149,si,Chl-a,0.0087,mg/l,0.3,DIA,GEMS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (740431.856 6476791.017),NaT,2016-11-22 10:00:00
1,NaN,URY00149,si,Chl-a,0.0260,mg/l,0.3,DIA,GEMS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (740431.856 6476791.017),NaT,2018-01-15 15:05:00
2,NaN,URY00149,si,Chl-a,0.0094,mg/l,0.3,DIA,GEMS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (740431.856 6476791.017),NaT,2018-04-09 14:24:00
3,NaN,URY00149,si,Chl-a,0.0029,mg/l,0.3,DIA,GEMS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (740431.856 6476791.017),NaT,2019-03-18 16:25:00
4,NaN,URY00149,si,Chl-a,0.0023,mg/l,0.3,DIA,GEMS,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (740431.856 6476791.017),NaT,2019-06-24 15:05:00


In [30]:
union_completo.to_file("../data/data/chla_oan_gems_union.geojson", driver="GeoJSON")